# Cleaning Base Dataset

## Imputing missing data

In [2]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from Func import *

In [3]:
np.set_printoptions(precision=2, suppress=True)

### Parameters

In [4]:
K = 7
m = 28722

In [5]:
df, encoders = load_dataframe("./data/patient_dataset.xlsx")
df = df.iloc[:m].copy()
print(df.columns)
print(df.head())

Index(['Small Cell', 'DxDate', 'AgeDx', 'Sex', 'Race1', 'Hispanic',
       'ComboSEERStage', 'SmokingHistory', 'Urban_Rural_ARHA', 'Radon',
       'CoCAccreditedFlag', 'DateLastContact', 'VitalStatus'],
      dtype='object')
   Small Cell    DxDate  AgeDx  Sex  Race1  Hispanic  ComboSEERStage  \
0           0  20170623      1    1      2         0               0   
1           0  20170516      2    0      0         0               0   
2           0  20180413      2    1      0         0               0   
3           0  20191119      3    1      0         0               2   
4           0  20200904      3    1      2         0               0   

   SmokingHistory  Urban_Rural_ARHA  Radon  CoCAccreditedFlag  \
0               0                 0      1                  1   
1               1                 0      1                  1   
2              -1                 0      1                  1   
3              -1                 0      1                  0   
4              -1

In [6]:
for col, mapping in encoders.items():
    print(f"{col}: {mapping}")

Small Cell: {'No': 0, 'Yes': 1}
Sex: {'Female': 0, 'Male': 1, 'Other': 2}
Race1: {'Black': 0, 'Other': 1, 'White': 2}
Hispanic: {'N': 0, 'Y': 1}
ComboSEERStage: {'Distant': 0, 'In Situ': 1, 'Localized': 2, 'Regional': 3}
SmokingHistory: {'Current': 0, 'Former': 1, 'Never': 2}
Urban_Rural_ARHA: {'Rural': 0, 'Urban': 1}
Radon: {'Zone 1': 0, 'Zone 2': 1, 'Zone 3': 2}
CoCAccreditedFlag: {'No': 0, 'Yes': 1}
VitalStatus: {'Alive': 0, 'Dead': 1}


In [7]:
X = []
for i in range(len(df)):
    V = [df.iloc[i]['Small Cell'],
         df.iloc[i]['AgeDx'],
         df.iloc[i]['Sex'],
         df.iloc[i]['Race1'],
         df.iloc[i]['Hispanic'],
         df.iloc[i]['ComboSEERStage'],
         df.iloc[i]['SmokingHistory'],
         df.iloc[i]['Urban_Rural_ARHA'],
         df.iloc[i]['Radon'],
        ]
    V = [int(val) for val in V]
    X.append(V)

In [ ]:
X = np.array(X)
S = patient_similarity_matrix(X)
print(S[:5, :5])

# Exports numpy object to protect against crashes
np.save("patient_similarity_matrix.npy", S)

In [ ]:
# Find k similar neighbors for each patient

# Loads from numpy object
S = np.load("patient_similarity_matrix.npy", mmap_mode="r")
Nbr = find_knn(S, K)
print(Nbr)

In [ ]:
df = impute_dataframe(df, S, Nbr)
print(df.head())

In [ ]:
# Rename imputed spreadsheets
df_completed = impute_dataframe(df, S, Nbr)
df_completed = decode_dataframe(df_completed, encoders)
print(df_completed.head())

In [ ]:
df.to_excel("./data/patients_imputed.xlsx", index=False)
df_completed.to_excel("./data/patients_imputed_renamed.xlsx", index=False)

## General dataframe cleanup

In [8]:
df = pd.read_excel("./data/patients_imputed_renamed.xlsx")
df = df.rename(columns={"Small Cell": "SmallCell", "Race1": "Race", "ComboSEERStage": "Stage", "Urban_Rural_ARHA": "Rurality", "CoCAccreditedFlag": "CoC"}).copy()
df = df[df["Sex"] != "Other"].copy()
df["AliveFlag"] = (df["VitalStatus"].map({"Dead": 0, "Alive": 1})).copy()
df

,SmallCell,DxDate,AgeDx,Sex,Race,Hispanic,Stage,SmokingHistory,Rurality,Radon,CoC,DateLastContact,VitalStatus,AliveFlag
0,No,20170623,1,Male,White,N,Distant,Current,Rural,Zone 2,Yes,20171228.0,Dead,0
1,No,20170516,2,Female,Black,N,Distant,Former,Rural,Zone 2,Yes,20171110.0,Alive,1
2,No,20180413,2,Male,Black,N,Distant,Current,Rural,Zone 2,Yes,20190108.0,Alive,1
3,No,20191119,3,Male,Black,N,Localized,Current,Rural,Zone 2,No,20200627.0,Dead,0
4,No,20200904,3,Male,White,N,Distant,Former,Rural,Zone 2,No,20200916.0,Dead,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28715,No,20220718,3,Female,White,N,Distant,Current,Rural,Zone 2,No,20220718.0,Dead,0
28716,No,20211115,2,Female,White,N,Distant,Former,Rural,Zone 2,Yes,20231228.0,Alive,1
28717,No,202310,2,Female,Black,N,Distant,Current,Rural,Zone 3,Yes,20240923.0,Dead,0
28718,Yes,20230203,3,Female,White,N,Localized,Current,Rural,Zone 2,No,20230908.0,Dead,0


In [10]:
def parse_date(value):
    """
    Converts dates of the form YYYYMMDD, YYYYMM, and YYYY to Pandas datetime objects.

    Parameters:
    values : string
        String of the form YYYYMMDD, YYYYMM, or YYYY.

    Returns:
    Pandas datetime object of the relevant form, otherwise returns Not A Time (NAT) object.
    """
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip().replace(".0", "")

    if len(value) == 8:
        return pd.to_datetime(value, format="%Y%m%d", errors="coerce")
    elif len(value) == 6:
        return pd.to_datetime(value, format="%Y%m", errors="coerce")
    elif len(value) == 4:
        return pd.to_datetime(value, format="%Y", errors="coerce")
    return pd.NaT

In [11]:
df["DxDateParsed"] = df["DxDate"].apply(parse_date)
df["DateLastContactParsed"] = df["DateLastContact"].apply(parse_date)

In [12]:
df = df.dropna().copy()
df

,SmallCell,DxDate,AgeDx,Sex,Race,Hispanic,Stage,SmokingHistory,Rurality,Radon,CoC,DateLastContact,VitalStatus,AliveFlag,DxDateParsed,DateLastContactParsed
0,No,20170623,1,Male,White,N,Distant,Current,Rural,Zone 2,Yes,20171228.0,Dead,0,2017-06-23,2017-12-28
1,No,20170516,2,Female,Black,N,Distant,Former,Rural,Zone 2,Yes,20171110.0,Alive,1,2017-05-16,2017-11-10
2,No,20180413,2,Male,Black,N,Distant,Current,Rural,Zone 2,Yes,20190108.0,Alive,1,2018-04-13,2019-01-08
3,No,20191119,3,Male,Black,N,Localized,Current,Rural,Zone 2,No,20200627.0,Dead,0,2019-11-19,2020-06-27
4,No,20200904,3,Male,White,N,Distant,Former,Rural,Zone 2,No,20200916.0,Dead,0,2020-09-04,2020-09-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28715,No,20220718,3,Female,White,N,Distant,Current,Rural,Zone 2,No,20220718.0,Dead,0,2022-07-18,2022-07-18
28716,No,20211115,2,Female,White,N,Distant,Former,Rural,Zone 2,Yes,20231228.0,Alive,1,2021-11-15,2023-12-28
28717,No,202310,2,Female,Black,N,Distant,Current,Rural,Zone 3,Yes,20240923.0,Dead,0,2023-10-01,2024-09-23
28718,Yes,20230203,3,Female,White,N,Localized,Current,Rural,Zone 2,No,20230908.0,Dead,0,2023-02-03,2023-09-08


## Defining pre- and post-lockdown temporal cohorts

In [13]:
df["Period"] = np.select(
    [
        df["DxDateParsed"].le(pd.Timestamp("2020-03-13")),
        df["DxDateParsed"].ge(pd.Timestamp("2021-05-31"))
    ],
    [
        "Pre-COVID",
        "Post-COVID"
    ],
    default="Transition"
)

In [14]:
pre_df = df[df["Period"] == "Pre-COVID"].copy()
post_df = df[df["Period"] == "Post-COVID"].copy()
mid_df = df[df["Period"] == "Transition"].copy()

In [15]:
pre_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13330 entries, 0 to 28707
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   SmallCell              13330 non-null  object        
 1   DxDate                 13330 non-null  int64         
 2   AgeDx                  13330 non-null  int64         
 3   Sex                    13330 non-null  object        
 4   Race                   13330 non-null  object        
 5   Hispanic               13330 non-null  object        
 6   Stage                  13330 non-null  object        
 7   SmokingHistory         13330 non-null  object        
 8   Rurality               13330 non-null  object        
 9   Radon                  13330 non-null  object        
 10  CoC                    13330 non-null  object        
 11  DateLastContact        13330 non-null  float64       
 12  VitalStatus            13330 non-null  object        
 13  AliveF

# Fitting Logistic Regression Models

In [16]:
formula = """
AliveFlag ~ C(SmallCell, Treatment(reference="No"))
    + C(AgeDx, Treatment(reference=2))
    + C(Sex, Treatment(reference="Female"))
    + C(Race, Treatment(reference="Black"))
    + C(Stage, Treatment(reference="Distant"))
    + C(SmokingHistory, Treatment(reference="Current"))
    + C(Rurality, Treatment(reference="Rural"))
    + C(Radon, Treatment(reference="Zone 1"))
    + C(CoC, Treatment(reference="No"))
"""

## Pre-COVID

In [17]:
pre_model = smf.logit(formula=formula, data=pre_df).fit()
print(pre_model.summary())

Optimization terminated successfully.
         Current function value: 0.338194
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:              AliveFlag   No. Observations:                13330
Model:                          Logit   Df Residuals:                    13312
Method:                           MLE   Df Model:                           17
Date:                Tue, 11 Aug 2026   Pseudo R-squ.:                  0.1510
Time:                        21:23:12   Log-Likelihood:                -4508.1
converged:                       True   LL-Null:                       -5309.9
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------------------------
Intercept 

In [18]:
pre_results = pd.DataFrame({
    "Coefficient": pre_model.params,
    "Odds Ratio": np.exp(pre_model.params),
    "CI Lower": np.exp(pre_model.conf_int()[0]),
    "CI Upper": np.exp(pre_model.conf_int()[1]),
    "p-value": pre_model.pvalues
})

pre_results

,Coefficient,Odds Ratio,CI Lower,CI Upper,p-value
Intercept,-2.966478,0.051484,0.041041,0.064584,4.500928e-145
"C(SmallCell, Treatment(reference=""No""))[T.Yes]",-0.798329,0.450080,0.358110,0.565670,7.640400e-12
"C(AgeDx, Treatment(reference=2))[T.0]",2.586265,13.280075,1.426599,123.622977,2.308060e-02
"C(AgeDx, Treatment(reference=2))[T.1]",1.039542,2.827921,2.198550,3.637459,5.801930e-16
"C(AgeDx, Treatment(reference=2))[T.3]",-1.113253,0.328489,0.285078,0.378509,1.792789e-53
"C(Sex, Treatment(reference=""Female""))[T.Male]",-0.457832,0.632654,0.567932,0.704751,9.197085e-17
"C(Race, Treatment(reference=""Black""))[T.Other]",1.000387,2.719333,1.691013,4.372983,3.670085e-05
"C(Race, Treatment(reference=""Black""))[T.White]",0.111444,1.117891,0.968921,1.289765,1.266909e-01
"C(Stage, Treatment(reference=""Distant""))[T.In Situ]",2.330580,10.283900,4.892195,21.617823,7.828904e-10
"C(Stage, Treatment(reference=""Distant""))[T.Localized]",1.961206,7.107893,6.156096,8.206846,1.715711e-157


## Post-COVID

In [19]:
post_model = smf.logit(formula=formula, data=post_df).fit()
print(post_model.summary())

Optimization terminated successfully.
         Current function value: 0.529753
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:              AliveFlag   No. Observations:                10616
Model:                          Logit   Df Residuals:                    10598
Method:                           MLE   Df Model:                           17
Date:                Tue, 11 Aug 2026   Pseudo R-squ.:                  0.1472
Time:                        21:23:19   Log-Likelihood:                -5623.9
converged:                       True   LL-Null:                       -6594.6
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------------------------
Intercept 

In [20]:
post_results = pd.DataFrame({
    "Coefficient": post_model.params,
    "Odds Ratio": np.exp(post_model.params),
    "CI Lower": np.exp(post_model.conf_int()[0]),
    "CI Upper": np.exp(post_model.conf_int()[1]),
    "p-value": post_model.pvalues
})

post_results

,Coefficient,Odds Ratio,CI Lower,CI Upper,p-value
Intercept,-1.533887,0.215696,0.180011,0.258454,4.718540e-62
"C(SmallCell, Treatment(reference=""No""))[T.Yes]",-0.822525,0.439321,0.370327,0.521169,3.863772e-21
"C(AgeDx, Treatment(reference=2))[T.0]",1.285970,3.618178,0.623303,21.002976,1.518176e-01
"C(AgeDx, Treatment(reference=2))[T.1]",0.553911,1.740044,1.321082,2.291875,8.107052e-05
"C(AgeDx, Treatment(reference=2))[T.3]",-0.686653,0.503258,0.453672,0.558264,1.710484e-38
"C(Sex, Treatment(reference=""Female""))[T.Male]",-0.345413,0.707928,0.645831,0.775994,1.650526e-13
"C(Race, Treatment(reference=""Black""))[T.Other]",0.673834,1.961743,1.217958,3.159744,5.593066e-03
"C(Race, Treatment(reference=""Black""))[T.White]",-0.086288,0.917330,0.817923,1.028818,1.403516e-01
"C(Stage, Treatment(reference=""Distant""))[T.In Situ]",1.163034,3.199626,1.441775,7.100695,4.242600e-03
"C(Stage, Treatment(reference=""Distant""))[T.Localized]",1.846522,6.337739,5.676608,7.075868,1.107039e-236


## Mid-COVID

In [21]:
mid_model = smf.logit(formula=formula, data=mid_df).fit()
print(mid_model.summary())

Optimization terminated successfully.
         Current function value: 0.402790
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:              AliveFlag   No. Observations:                 4724
Model:                          Logit   Df Residuals:                     4706
Method:                           MLE   Df Model:                           17
Date:                Tue, 11 Aug 2026   Pseudo R-squ.:                  0.1418
Time:                        21:23:23   Log-Likelihood:                -1902.8
converged:                       True   LL-Null:                       -2217.3
Covariance Type:            nonrobust   LLR p-value:                1.052e-122
                                                                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------------------------
Intercept 

In [22]:
mid_results = pd.DataFrame({
    "Coefficient": mid_model.params,
    "Odds Ratio": np.exp(mid_model.params),
    "CI Lower": np.exp(mid_model.conf_int()[0]),
    "CI Upper": np.exp(mid_model.conf_int()[1]),
    "p-value": mid_model.pvalues
})

mid_results

,Coefficient,Odds Ratio,CI Lower,CI Upper,p-value
Intercept,-2.007752,0.134290,0.098226,0.183595,2.612744e-36
"C(SmallCell, Treatment(reference=""No""))[T.Yes]",-1.030106,0.356969,0.249712,0.510296,1.604764e-08
"C(AgeDx, Treatment(reference=2))[T.0]",2.406914,11.099650,1.016406,121.213606,4.846080e-02
"C(AgeDx, Treatment(reference=2))[T.1]",0.218943,1.244760,0.753533,2.056218,3.925795e-01
"C(AgeDx, Treatment(reference=2))[T.3]",-0.792221,0.452838,0.372566,0.550405,1.751727e-15
"C(Sex, Treatment(reference=""Female""))[T.Male]",-0.459602,0.631535,0.536628,0.743226,3.173567e-08
"C(Race, Treatment(reference=""Black""))[T.Other]",1.084664,2.958444,1.406249,6.223926,4.257788e-03
"C(Race, Treatment(reference=""Black""))[T.White]",-0.180998,0.834437,0.682697,1.019904,7.714420e-02
"C(Stage, Treatment(reference=""Distant""))[T.In Situ]",2.730627,15.342500,5.116453,46.006931,1.096275e-06
"C(Stage, Treatment(reference=""Distant""))[T.Localized]",1.652267,5.218800,4.278954,6.365076,8.455106e-60


# Calculating Wald Test

In [23]:
from Wald import wald_test_or

-4.623673439938184 3.770032323435757e-06


In [24]:
def wald_frame(pre_model, post_model):
    """
    Compares coefficients from pre- and post-COVID logistic regression models using wald_test_or() from Wald.py.
    """
    # Same coefficients in both models
    terms = pre_model.params.index

    # Drops intercept
    terms = terms.drop("Intercept", errors="ignore")

    rows = []

    for term in terms:
        beta_pre = pre_model.params[term]
        beta_post = post_model.params[term]

        se_pre = pre_model.bse[term]
        se_post = post_model.bse[term]

        or_pre = np.exp(beta_pre)
        or_post = np.exp(beta_post)

        z_stat, p_value = wald_test_or(OR_pre=or_pre, OR_post=or_post, SE_pre=se_pre, SE_post=se_post)

        rows.append({
            "Predictor": term,
            "Beta Pre": beta_pre,
            "OR Pre": or_pre,
            "SE Pre": se_pre,
            "Beta Post": beta_post,
            "OR Post": or_post,
            "SE Post": se_post,
            "Wald Z": z_stat,
            "p-value": p_value,
            "Significant": p_value < 0.05
        })

    return pd.DataFrame(rows)

In [26]:
wald_results = wald_frame(pre_model, post_model)
wald_results

,Predictor,Beta Pre,OR Pre,SE Pre,Beta Post,OR Post,SE Post,Wald Z,p-value,Significant
0,"C(SmallCell, Treatment(reference=""No""))[T.Yes]",-0.798329,0.450080,0.116627,-0.822525,0.439321,0.087167,0.166174,0.868020,False
1,"C(AgeDx, Treatment(reference=2))[T.0]",2.586265,13.280075,1.138272,1.285970,3.618178,0.897309,0.897112,0.369659,False
2,"C(AgeDx, Treatment(reference=2))[T.1]",1.039542,2.827921,0.128443,0.553911,1.740044,0.140543,2.550660,0.010752,True
3,"C(AgeDx, Treatment(reference=2))[T.3]",-1.113253,0.328489,0.072317,-0.686653,0.503258,0.052924,-4.760426,0.000002,True
4,"C(Sex, Treatment(reference=""Female""))[T.Male]",-0.457832,0.632654,0.055063,-0.345413,0.707928,0.046839,-1.555103,0.119922,False
5,"C(Race, Treatment(reference=""Black""))[T.Other]",1.000387,2.719333,0.242381,0.673834,1.961743,0.243197,0.951062,0.341573,False
6,"C(Race, Treatment(reference=""Black""))[T.White]",0.111444,1.117891,0.072969,-0.086288,0.917330,0.058521,2.113948,0.034520,True
7,"C(Stage, Treatment(reference=""Distant""))[T.In ...",2.330580,10.283900,0.379057,1.163034,3.199626,0.406721,2.100003,0.035729,True
8,"C(Stage, Treatment(reference=""Distant""))[T.Loc...",1.961206,7.107893,0.073350,1.846522,6.337739,0.056209,1.241028,0.214596,False
9,"C(Stage, Treatment(reference=""Distant""))[T.Reg...",1.285326,3.615847,0.079140,1.161093,3.193421,0.061204,1.241767,0.214322,False


# Writing to Excel

In [ ]:
output_file = "./data/logistic_regression_results.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    
    # Write statsmodels summary tables
    pre_model.summary2().tables[0].to_excel(
        writer,
        sheet_name="Pre Model Data",
        index=False,
        header=False
    )

    pre_model.summary2().tables[1].to_excel(
        writer,
        sheet_name="Pre Coefficients"
    )

    mid_model.summary2().tables[0].to_excel(
        writer,
        sheet_name="Mid Model Data",
        index=False,
        header=False
    )

    mid_model.summary2().tables[1].to_excel(
        writer,
        sheet_name="Mid Coefficients"
    )

    post_model.summary2().tables[0].to_excel(
        writer,
        sheet_name="Post Model Data",
        index=False,
        header=False
    )

    post_model.summary2().tables[1].to_excel(
        writer,
        sheet_name="Post Coefficients"
    )

    # Write calculated result tables
    pre_results.to_excel(
        writer,
        sheet_name="Pre Results",
        index=True
    )

    mid_results.to_excel(
        writer,
        sheet_name="Mid Results",
        index=True
    )

    post_results.to_excel(
        writer,
        sheet_name="Post Results",
        index=True
    )

    wald_results.to_excel(
        writer,
        sheet_name="Wald Results",
        index=False
    )
    

# Generating Pairwise Correlation Matrix for Age

## Full Dataset

In [27]:
corr_features = [
    "AgeDx",
    "SmallCell",
    "Sex",
    "Race",
    "Stage",
    "SmokingHistory",
    "Rurality",
    "Radon",
    "CoC"
]


corr_df = df[corr_features].copy()

In [28]:
corr_encoded_df = pd.get_dummies(
    corr_df,
    columns=[
        "SmallCell",
        "Sex",
        "Race",
        "Stage",
        "SmokingHistory",
        "Rurality",
        "Radon",
        "CoC"
    ],
    drop_first=False,
    dtype=int
)

In [29]:
age_corr = (corr_encoded_df.corr(method="spearman")["AgeDx"].drop("AgeDx").sort_values(key=abs, ascending=False))
age_corr

Race_White                0.082691
Race_Black               -0.079718
SmokingHistory_Former     0.077034
Stage_Localized           0.072665
Stage_Distant            -0.066716
SmokingHistory_Current   -0.065908
SmallCell_Yes            -0.052498
SmallCell_No              0.052498
CoC_No                    0.045958
CoC_Yes                  -0.045958
SmokingHistory_Never      0.041998
Radon_Zone 2             -0.026122
Radon_Zone 1              0.019529
Race_Other               -0.018683
SmokingHistory_Unk       -0.017388
Sex_Male                 -0.012876
Sex_Female                0.012876
Rurality_Rural           -0.012767
Rurality_Urban            0.012767
Radon_Zone 3              0.006943
Stage_In Situ             0.005915
Stage_Regional           -0.000012
Name: AgeDx, dtype: float64

## Pre-COVID

In [31]:
pre_corr_df = pre_df[corr_features].copy()

In [32]:
pre_corr_encoded_df = pd.get_dummies(
    pre_corr_df,
    columns=[
        "SmallCell",
        "Sex",
        "Race",
        "Stage",
        "SmokingHistory",
        "Rurality",
        "Radon",
        "CoC"
    ],
    drop_first=False,
    dtype=int
)

In [33]:
pre_age_corr = (pre_corr_encoded_df.corr(method="spearman")["AgeDx"].drop("AgeDx").sort_values(key=abs, ascending=False))
pre_age_corr

SmokingHistory_Former     0.096944
SmokingHistory_Current   -0.085153
Stage_Localized           0.080612
Race_White                0.074675
Race_Black               -0.071254
Stage_Distant            -0.067092
SmallCell_Yes            -0.058113
SmallCell_No              0.058113
CoC_No                    0.050401
CoC_Yes                  -0.050401
SmokingHistory_Never      0.034336
Radon_Zone 2             -0.025976
Race_Other               -0.020003
Rurality_Rural           -0.019061
Rurality_Urban            0.019061
Radon_Zone 1              0.019046
SmokingHistory_Unk       -0.009433
Stage_In Situ             0.007678
Stage_Regional           -0.007443
Radon_Zone 3              0.007301
Sex_Male                 -0.003570
Sex_Female                0.003570
Name: AgeDx, dtype: float64

## Post-COVID

In [34]:
post_corr_df = post_df[corr_features].copy()

In [35]:
post_corr_encoded_df = pd.get_dummies(
    post_corr_df,
    columns=[
        "SmallCell",
        "Sex",
        "Race",
        "Stage",
        "SmokingHistory",
        "Rurality",
        "Radon",
        "CoC"
    ],
    drop_first=False,
    dtype=int
)

In [36]:
post_age_corr = (post_corr_encoded_df.corr(method="spearman")["AgeDx"].drop("AgeDx").sort_values(key=abs, ascending=False))
post_age_corr

Race_White                0.094807
Race_Black               -0.093318
Stage_Localized           0.067600
Stage_Distant            -0.063816
SmokingHistory_Former     0.060566
SmokingHistory_Current   -0.054930
SmallCell_Yes            -0.046004
SmallCell_No              0.046004
SmokingHistory_Never      0.044631
Radon_Zone 2             -0.038440
CoC_No                    0.036377
CoC_Yes                  -0.036377
Radon_Zone 1              0.034582
Sex_Male                 -0.019683
Sex_Female                0.019683
SmokingHistory_Unk       -0.019436
Race_Other               -0.012926
Stage_In Situ             0.006719
Radon_Zone 3              0.003774
Rurality_Rural           -0.002321
Rurality_Urban            0.002321
Stage_Regional            0.000146
Name: AgeDx, dtype: float64